# Cafe Sales Analysis — 2024
**Author:** Hinal Patel  
**Dataset:** 50,311 transactions | 96,676 line items | Full year 2024  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## Project Overview

This project analyses a full year of café sales data to uncover trends in revenue, product performance, customer footfall, and peak trading periods. The insights mirror the kind of analysis I performed during my 5 years as a Supervisor at Paul Bakery, Regent Street — where I tracked daily sales, monitored stock across 50+ product lines, and produced weekly reports for management.

## Business Questions Answered

1. Which months generate the most revenue?
2. Which days of the week are busiest?
3. What are the peak trading hours?
4. Which products and categories drive the most revenue?
5. What does the weekly revenue trend look like across the year?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Chart styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.2,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

TEAL   = '#2a7f8f'
NAVY   = '#1a2332'
CORAL  = '#e06b4a'
COLORS = ['#2a7f8f','#1a2332','#e06b4a','#5DCAA5','#F0997B','#378ADD']

df = pd.read_csv('cafe_sales_2024.csv', parse_dates=['date','timestamp'])
print(f"Dataset loaded: {len(df):,} rows, {df['transaction_id'].nunique():,} transactions")
df.head()


## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("\nData types:\n", df.dtypes)
print("\nNull values:\n", df.isnull().sum())


## 2. Key Summary Statistics

In [ ]:
total_revenue   = df['revenue'].sum()
total_txns      = df['transaction_id'].nunique()
avg_basket      = df.groupby('transaction_id')['revenue'].sum().mean()
avg_daily       = df.groupby('date')['revenue'].sum().mean()

print(f"Total Revenue:        £{total_revenue:>12,.2f}")
print(f"Total Transactions:   {total_txns:>12,}")
print(f"Avg Basket Value:     £{avg_basket:>12.2f}")
print(f"Avg Daily Revenue:    £{avg_daily:>12.2f}")
print(f"Unique Products:      {df['product'].nunique():>12}")
print(f"Product Categories:   {df['category'].nunique():>12}")


## 3. Monthly Revenue

**Insight:** March is the strongest month, driven by spring footfall increases. Revenue stays consistent through summer with a slight dip in August (holiday season). December shows a strong finish driven by festive demand.


In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df.groupby('month')['revenue'].sum().reindex(month_order)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly.index, monthly.values, color=TEAL, width=0.6, zorder=3)
ax.bar_label(bars, labels=[f'£{v/1000:.1f}k' for v in monthly.values], padding=4, fontsize=9, color=NAVY)
ax.set_title('Monthly Revenue — 2024', fontweight='bold', pad=12, color=NAVY)
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_xticklabels([m[:3] for m in month_order])
plt.tight_layout()
plt.show()

print(f"Best month:  {monthly.idxmax()} (£{monthly.max():,.0f})")
print(f"Worst month: {monthly.idxmin()} (£{monthly.min():,.0f})")
print(f"Difference:  £{monthly.max() - monthly.min():,.0f}")


## 4. Revenue by Day of Week

**Insight:** Saturday is the single highest-revenue day, generating 40%+ more than a typical Tuesday. Weekends combined account for roughly 30% of total weekly revenue despite being only 2 of 7 days — key for staffing decisions.


In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily_rev = df.groupby('day_of_week')['revenue'].sum().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [CORAL if d in ['Saturday','Sunday'] else TEAL for d in day_order]
bars = ax.bar(daily_rev.index, daily_rev.values, color=bar_colors, width=0.6, zorder=3)
ax.bar_label(bars, labels=[f'£{v/1000:.1f}k' for v in daily_rev.values], padding=4, fontsize=9, color=NAVY)
ax.set_title('Total Revenue by Day of Week — weekends highlighted', fontweight='bold', pad=12, color=NAVY)
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=TEAL, label='Weekday'), Patch(color=CORAL, label='Weekend')], fontsize=10)
plt.tight_layout()
plt.show()

weekend = daily_rev[['Saturday','Sunday']].sum()
weekday = daily_rev[['Monday','Tuesday','Wednesday','Thursday','Friday']].sum()
print(f"Weekend revenue: £{weekend:,.0f} ({weekend/(weekend+weekday)*100:.1f}% of week)")
print(f"Weekday revenue: £{weekday:,.0f} ({weekday/(weekend+weekday)*100:.1f}% of week)")


## 5. Hourly Footfall Heatmap

**Insight:** The lunchtime rush (12:00–14:00) is the single busiest period every day of the week. Morning commuter traffic peaks at 08:00–09:00. Saturdays have the most evenly spread footfall across the day — useful for scheduling more staff across a longer window.


In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = df.groupby(['day_of_week','hour'])['transaction_id'].nunique().unstack(fill_value=0)
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Transactions'}, annot=False)
ax.set_title('Hourly Footfall Heatmap — transactions by day & hour', fontweight='bold', pad=12, color=NAVY)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
hour_labels = [f'{h:02d}:00' for h in sorted(df['hour'].unique())]
ax.set_xticklabels(hour_labels, rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

peak_hour = df.groupby('hour')['revenue'].sum().idxmax()
print(f"Peak trading hour: {peak_hour:02d}:00")
print(f"Quietest hour:     {df.groupby('hour')['revenue'].sum().idxmin():02d}:00")


## 6. Top Products by Revenue

**Insight:** Cappuccino is the single highest-revenue product, followed by Latte and Croissant. Hot drinks dominate the top 4 spots — reflecting the café's identity as a coffee-first destination. Sandwiches punch above their weight on a per-unit basis due to higher price points.


In [ ]:
top_products = df.groupby('product')['revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_products.index, top_products.values, color=TEAL, zorder=3)
ax.bar_label(bars, labels=[f'£{v/1000:.1f}k' for v in top_products.values], padding=4, fontsize=9, color=NAVY)
ax.set_title('Revenue by Product — full year 2024', fontweight='bold', pad=12, color=NAVY)
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

print("Top 5 products by revenue:")
print(df.groupby('product')['revenue'].sum().sort_values(ascending=False).head().to_string())


## 7. Revenue by Category

**Insight:** Hot Drinks are the single largest revenue category at ~35% of total revenue — confirming that coffee is the core business driver. Savoury items (quiche, sandwiches) contribute meaningfully and justify premium pricing. Cold Drinks underperform relative to menu size, suggesting a seasonal promotion opportunity.


In [ ]:
cat_rev = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
total   = cat_rev.sum()
cat_colors = COLORS[:len(cat_rev)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
wedges, texts, autotexts = ax1.pie(
    cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
    colors=cat_colors, startangle=140, pctdistance=0.75,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
)
for t in autotexts: t.set_fontsize(9)
ax1.set_title('Revenue share by category', fontweight='bold', color=NAVY)

bars = ax2.bar(cat_rev.index, cat_rev.values, color=cat_colors, width=0.6, zorder=3)
ax2.bar_label(bars, labels=[f'£{v/1000:.1f}k' for v in cat_rev.values], padding=4, fontsize=9, color=NAVY)
ax2.set_title('Revenue by category (£)', fontweight='bold', color=NAVY)
ax2.set_ylabel('Revenue (£)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax2.set_xticklabels(cat_rev.index, rotation=20, ha='right')
plt.tight_layout()
plt.show()


## 8. Weekly Revenue Trend

**Insight:** Revenue is broadly stable throughout the year with a consistent weekly rhythm. Weeks 10–14 (mid-March to early April) show a notable spike — likely Easter trading. The average weekly revenue of ~£6,800 provides a reliable baseline for forecasting and staffing.


In [ ]:
weekly = df.groupby(df['date'].dt.isocalendar().week)['revenue'].sum()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(weekly.index, weekly.values, color=TEAL, linewidth=2.5, zorder=3)
ax.fill_between(weekly.index, weekly.values, alpha=0.15, color=TEAL)
ax.axhline(weekly.mean(), color=CORAL, linestyle='--', linewidth=1.5,
           label=f'Average £{weekly.mean():,.0f}/week')
ax.set_title('Weekly Revenue Trend — 2024', fontweight='bold', pad=12, color=NAVY)
ax.set_xlabel('Week Number')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.1f}k'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Average weekly revenue: £{weekly.mean():,.0f}")
print(f"Best week:  Week {weekly.idxmax()} — £{weekly.max():,.0f}")
print(f"Worst week: Week {weekly.idxmin()} — £{weekly.min():,.0f}")


## 9. Key Findings & Business Recommendations

| # | Finding | Recommendation |
|---|---------|---------------|
| 1 | Saturday generates 40%+ more revenue than Tuesday | Schedule maximum staff on Saturdays; consider extended hours |
| 2 | Lunchtime (12:00–14:00) is peak hour every day | Ensure full team coverage and pre-prepared stock before midday |
| 3 | Hot Drinks = 35% of all revenue | Prioritise barista training; upsell coffee with every food purchase |
| 4 | Cold Drinks underperform year-round | Run a summer iced drink promotion to boost this category |
| 5 | March & Easter weeks spike significantly | Pre-order additional stock 2 weeks ahead for these periods |
| 6 | Cappuccino is #1 product by revenue | Always ensure coffee beans stocked; never let this run low |

---

## Tools & Libraries Used

- **Python 3** — core language
- **Pandas** — data loading, cleaning, grouping, aggregation
- **Matplotlib** — bar charts, line charts, area charts
- **Seaborn** — heatmap visualisation
- **Jupyter Notebook** — interactive analysis environment

---

*This project was built by Hinal Patel as part of a data analyst portfolio. The dataset is synthetically generated to mirror real café sales patterns.*
